# CrewAI Agent with a Custom Tool

In [1]:
!pip install -q crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.8/820.8 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [2]:
import crewai
import crewai_tools

print(crewai.__version__)
print(crewai_tools.__version__)

1.15.10
1.15.10


# Set API Keys

In [3]:
from google.colab import userdata
import os
# os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_NEW')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


# Import Dependencies

In [5]:
from crewai import Agent, Crew, Task, Process, LLM, Process
from crewai_tools import SerperDevTool, DirectoryReadTool
from pydantic import BaseModel
from typing import List

# from dotenv import load_dotenv
# load_dotenv()

#------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")
#------------------------------------------------------------------------


# Define LLMs

In [6]:
# Create an LLM with a temperature of 0 to ensure deterministic outputs
# -----------
# Create LLM
# -----------

# Create an LLM with a temperature of 0 to ensure deterministic outputs
# OPENAI LLMs
# llm = LLM(
#          # model="gpt-5.4-mini",
#           model="gpt-5.4-nano",
#           base_url="https://api.openai.com/v1",
#           api_key = os.environ["OPENAI_API_KEY"],
#           temperature=0.2)

# GROQ hosted LLMs
llm = LLM(
     model="llama-3.3-70b-versatile",
     base_url="https://api.groq.com/openai/v1",
     api_key=os.environ["GROQ_API_KEY"],
     temperature=0.4)



# Define a Custom Tools

In [7]:
from typing import Type
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
#-------------------------------------------------------------------
# Define a custom input data type for the tool
class MyToolInput(BaseModel):
    """Input schema for CustomTool."""
    tool_message: str = Field(..., description="This is a text message.")
    tool_number: int = Field(..., description="This is a number.")
#-------------------------------------------------------------------
# Define the tool
class CustomTool(BaseTool):
    name: str = "Custom Tool to echo message."
    description: str = "This tool echos message. It's vital for echoing messages."
    args_schema: Type[BaseModel] = MyToolInput

    def _run(self, tool_message: str, tool_number: int) -> str:
        # Your tool's logic here
        print("\n Runnnig custom tool......")
        return f"Echoing message: {tool_message}. Long live {tool_number} years + eternity!"


# Define Agents

In [8]:
# Create an agent using the tool

agent = Agent(
    role="Echo Agent",
    goal="Echo back input using custom tool",
    backstory="You are expert in echoing messages",
    tools=[CustomTool()],
    llm=llm,
    verbose=True
)


# Define Tasks

In [9]:
task = Task(
    description="Echo the provided message {message} {number}.",
    expected_output="You need to echo the message returned by the tool.",
    agent=agent,
    verbose=True
)

# Define Crew (Orchestration Layer)

In [10]:
crew = Crew(
    agents=[agent],
    tasks=[task]
    )

# Run the Crew

In [11]:
result = await crew.kickoff_async(inputs={ "message": "Hello, India", "number": "1100"})
# result = await crew.kickoff_async(inputs={ "number": "1100", "message": "Hello, India"})

print(result)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Echo Agent                                                                                              │
│                                                                                                                 │
│  Task: Echo the provided message Hello, India 1100.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


 Runnnig custom tool......
Tool custom_tool_to_echo_message executed with result: Echoing message: Hello, India. Long live 1100 years + eternity!...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Echo Agent                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello, India. Long live 1100 years + eternity!                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hello, India. Long live 1100 years + eternity!


In [12]:
result = await crew.kickoff_async(inputs={ "number": "1100", "message": "Hello, India"})
print(result)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Echo Agent                                                                                              │
│                                                                                                                 │
│  Task: Echo the provided message Hello, India 1100.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


 Runnnig custom tool......
Tool custom_tool_to_echo_message executed with result: Echoing message: Hello, India. Long live 1100 years + eternity!...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Echo Agent                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello, India. Long live 1100 years + eternity!                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hello, India. Long live 1100 years + eternity!
